# ChromaDB: Vector Database
**ChromaDB** is an open-source vector database. Its primary job is to store and retrieve vector embeddings i.e. numerical representations of data like text, images, or audio. This makes it incredibly fast at finding items based on their semantic meaning (e.g., finding text that is "about happiness" rather than just text that "contains the word happiness").

https://github.com/chroma-core/chroma


In [32]:
!pip install chromadb sentence-transformers pandas -q


# CRUD in ChromaDB



In [33]:
# Create a collection

collection = client.create_collection(name="my_first_collection")

print("Collection 'my_first_collection' created.")

Collection 'my_first_collection' created.


## Add Documents (Create)

You must provide three things for each item you add:

documents: A list of the raw text you want to store.

metadatas: A list of dictionaries containing extra info (like categories, sources, etc.). This is crucial for filtering.

ids: A list of unique string IDs for each document.

In [34]:
# Add documents to the collection
collection.add(
    documents=[
        "The quick brown fox jumps over the lazy dog.",
        "A journey of a thousand miles begins with a single step.",
        "To be or not to be, that is the question.",
        "I think, therefore I am.",
        "Life is what happens when you're busy making other plans."
    ],
    metadatas=[
        {"source": "proverb", "type": "animal"},
        {"source": "proverb", "type": "philosophy"},
        {"source": "shakespeare", "type": "philosophy"},
        {"source": "descartes", "type": "philosophy"},
        {"source": "lennon", "type": "quote"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Successfully added 5 documents to the collection.")

Successfully added 5 documents to the collection.


## Query Documents (Read)

 You don't search for exact keywords; you search for concepts.

We'll provide a list of query texts, and Chroma will automatically:

Convert our query text into a vector.

Compare that vector to all the vectors in the collection.

Return the most similar documents.

In [55]:
# Query the collection
results = collection.query(
    query_texts=["A quote about life and plans"],
    n_results=2,  # Ask for the top 2 most similar results
    # include=["metadatas", "documents", "embeddings", "distances"]
)

print("\n--- Query Results ---")
print(results)


--- Query Results ---
{'ids': [['movie_996', 'movie_939']], 'embeddings': None, 'documents': [['When a doubting young boy takes an extraordinary train ride to the North Pole, he embarks on a journey of self-discovery that shows him that the wonder of life never fades for those who believe.', 'Brilliant student Jeff Chang has the most important interview of his life tomorrow.  But today is still his birthday, what starts off as a casual celebration with friends evolves into a night of debauchery that risks to derail his life plan.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'title': 'The Polar Express'}, {'title': '21 & Over'}]], 'distances': [[1.1139190196990967, 1.1496102809906006]]}


## Query with Metadata Filters
What if you want to find the most similar documents that also meet a specific criteria? This is where metadata filtering shines.

Let's search for "thoughts on existence" but only among documents with the type "philosophy".

In [36]:
# Query with a 'where' filter
filtered_results = collection.query(
    query_texts=["thoughts on existence"],
    n_results=2,
    where={"type": "philosophy"}  # Only search documents where type is 'philosophy'
)

print("\n--- Filtered Query Results (type='philosophy') ---")
print(filtered_results)


--- Filtered Query Results (type='philosophy') ---
{'ids': [['doc4', 'doc3']], 'embeddings': None, 'documents': [['I think, therefore I am.', 'To be or not to be, that is the question.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'descartes', 'type': 'philosophy'}, {'type': 'philosophy', 'source': 'shakespeare'}]], 'distances': [[1.352301836013794, 1.390721082687378]]}


## Update and Delete
You can also manage your data using the unique IDs.

In [37]:
# Update the metadata for doc1
print("\n--- Updating doc1 ---")
collection.update(
    ids=["doc1"],
    metadatas=[{"source": "fable", "type": "animal"}] # Change source from 'proverb' to 'fable'
)

# You can also 'upsert'
# 'upsert' will update if the ID exists, or create it if it doesn't.
collection.upsert(
    ids=["doc6"],
    documents=["The early bird catches the worm."],
    metadatas=[{"source": "proverb", "type": "general"}]
)
print("Updated doc1 and upserted doc6.")


--- Updating doc1 ---
Updated doc1 and upserted doc6.


You can retrieve specific documents by their ID.

In [38]:
# Get documents by ID
print("\n--- Getting docs by ID ---")
retrieved = collection.get(ids=["doc1", "doc3"])
print(retrieved)


--- Getting docs by ID ---
{'ids': ['doc1', 'doc3'], 'embeddings': None, 'documents': ['The quick brown fox jumps over the lazy dog.', 'To be or not to be, that is the question.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'fable', 'type': 'animal'}, {'source': 'shakespeare', 'type': 'philosophy'}]}


You can delete items by their ID or by a filter.

In [39]:
# Delete an item by ID
print("\n--- Deleting doc6 ---")
collection.delete(ids=["doc6"])

# Delete items using a where filter
print("Deleting all 'shakespeare' documents...")
collection.delete(where={"source": "shakespeare"})

# Check the count
count = collection.count()
print(f"Total documents remaining in collection: {count}")


--- Deleting doc6 ---
Deleting all 'shakespeare' documents...
Total documents remaining in collection: 4


If you want to delete an entire collection, you can do so easily.

In [40]:
print("\n--- Deleting collection ---")
client.delete_collection(name="my_first_collection")
print("Collection 'my_first_collection' deleted.")

# You can also reset the entire database (clears all collections)
# client.reset()


--- Deleting collection ---
Collection 'my_first_collection' deleted.


# Recommendation system using ChromaDb

In [41]:
# Download the dataset
!wget https://raw.githubusercontent.com/alura-cursos/introducao-a-data-science/aula3/aula3.1/tmdb_5000_movies.csv -O tmdb_movies.csv

--2025-11-15 09:41:50--  https://raw.githubusercontent.com/alura-cursos/introducao-a-data-science/aula3/aula3.1/tmdb_5000_movies.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5698602 (5.4M) [text/plain]
Saving to: ‘tmdb_movies.csv’

tmdb_movies.csv     100%[===================>]   5.43M  --.-KB/s    in 0.02s   

2025-11-15 09:41:50 (299 MB/s) - ‘tmdb_movies.csv’ saved [5698602/5698602]



In [42]:
import pandas as pd

# Load the data
df = pd.read_csv('tmdb_movies.csv')

# --- Data Cleaning ---
# 1. We only need 'title' and 'overview'
df = df[['title', 'overview']]

# 2. Drop any movies with missing overviews
df = df.dropna(subset=['overview'])

# 3. (Optional) This dataset is big. Let's sample 1000 movies for a fast demo.
# For a real project, you'd use all 4800+.
df_sample = df.sample(n=1000, random_state=42)

print(f"Original data shape: {df.shape}")
print(f"Sampled data shape: {df_sample.shape}")

# Look at our data
df_sample.head()

Original data shape: (4800, 2)
Sampled data shape: (1000, 2)


,title,overview
596,I Spy,"When the Switchblade, the most sophisticated p..."
3371,Who's Your Caddy?,"When ""street smart"" rapper Christopher ""C-Note..."
3049,Sleepover,As their first year of high school looms ahead...
2909,The Cry of the Owl,A young woman becomes inexplicably attracted t...
8,Harry Potter and the Half-Blood Prince,"As Harry begins his sixth year at Hogwarts, he..."


In [52]:
import chromadb
from chromadb.utils import embedding_functions

# 1. Define the embedding model
# This will download the model from Hugging Face
model_name = "all-MiniLM-L6-v2"
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

# 2. Initialize the Chroma client
client = chromadb.Client() # We'll use an in-memory client for this demo

# 3. Create the collection
# We must specify our embedding function.
# The 'l2' distance is standard for many sentence-transformer models.
collection = client.get_or_create_collection(
    name="movie_recommendations",
    embedding_function=embedding_func,
    metadata={"hnsw:space": "l2"} # Use 'l2' (Euclidean) distance
)

# 4. Prepare data for Chroma
# We need lists of documents, metadatas, and unique IDs

documents = df_sample['overview'].tolist()
metadatas = [{'title': title} for title in df_sample['title'].tolist()]
ids = [f"movie_{i}" for i in range(len(documents))] # Create simple unique IDs

# 5. Add to the collection (This will take 1-2 minutes)
# Chroma will automatically:
# - Take each 'document' (the overview)
# - Run it through the 'embedding_func' to get a 384-dimension vector
# - Store that vector along with its metadata and ID
print("Starting to embed and add documents... (This may take a minute)")
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("All movies embedded and stored in the vector database!")

Starting to embed and add documents... (This may take a minute)
All movies embedded and stored in the vector database!


--- Recommendations for: "The Dark Knight" ---
-------------------------------------------------
1. All Superheroes Must Die
2. The Glimmer Man
3. Firestorm
4. Exit Wounds
5. Mad Max


In [54]:
_ = get_recommendations("Interstellar")

--- Recommendations for: "Interstellar" ---
-------------------------------------------------
1. Prometheus
2. Journey to the Center of the Earth
3. Treasure Planet
4. You Only Live Twice
5. Enter Nowhere


## Use Cases
Semantic Search: Improve search engines by finding documents that are similar in meaning to a query.

Recommendation Systems: Suggest items based on vector similarity such as recommending products or content.

Retrieval-Augmented Generation (RAG): Provide context to language models by retrieving relevant documents to answer questions.

Anomaly Detection: Identify outliers by comparing vector embeddings to a known distribution.